# TSF on OpenNeuro Psychedelic Datasets (LSD / Psilocybin / Ketamine)

**Datasets:**
- **LSD** — Carhart-Harris et al., OpenNeuro `ds003059` (fMRI). 
- **Psilocybin** — Precision Functional Mapping, OpenNeuro `ds006072` (fMRI). 
- **Ketamine** — NIMH Mechanism of Action Study, OpenNeuro `ds005917` (fMRI/MEG/EEG where available).

This notebook will:
1. **Download** the selected dataset subset via `openneuro-py` (limits to one subject by default).
2. **Locate BOLD** runs and extract ROI time series with `nilearn` using the Schaefer-200 atlas.
3. **Compute wavelet chronon features** and build a **cross-scale coupling** matrix with **zero diagonal**.
4. **Fit TSF kernel parameters** \((c_0, \beta, \gamma, \alpha_{ord})\) and estimate **\(\lambda\)**.
5. Save figures + JSON to `tsf_outputs/`.

**Tip:** Start with defaults (single subject) to verify everything runs, then expand SUBJECTS / MAX_FILES.


In [ ]:
# --- User parameters ---
DATASET = 'LSD'      # options: 'LSD', 'PSILO', 'KETAMINE'
SUBJECTS = ['01']    # BIDS subject labels (strings, zero-padded)
MAX_FILES = 4        # max number of BOLD files to process (keeps runtime sane)
TRIM_N_VOLS = 5      # drop initial volumes per run (stabilization)
ATLAS = 'schaefer_200' # atlas for time series extraction
N_SHELLS = 24        # wavelet scales / shells
WAVELET_NAME = 'morl'
RANDOM_STATE = 42

print({k:v for k,v in list(globals().items()) if k in ['DATASET','SUBJECTS','MAX_FILES','TRIM_N_VOLS','ATLAS','N_SHELLS','WAVELET_NAME']})

In [ ]:
# Install and import
import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split('==')[0])
    except Exception:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

pip_install('openneuro-py')
pip_install('nilearn')
pip_install('nibabel')
pip_install('pywavelets')
pip_install('numpy')
pip_install('scipy')
pip_install('pandas')
pip_install('matplotlib')

import os, re, json, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import pywt
from scipy.optimize import minimize
from nilearn import datasets, image, input_data
import openneuro as on

np.random.seed(RANDOM_STATE)
OUT = Path('tsf_outputs'); OUT.mkdir(exist_ok=True)
DATA_DIR = Path('data/openneuro'); DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Environment ready.")

## 1) Download dataset subset
We restrict to a single subject by default to keep the download small. Adjust `SUBJECTS` to expand.

In [ ]:
def dataset_id(name):
    return {'LSD':'ds003059', 'PSILO':'ds006072', 'KETAMINE':'ds005917'}[name]

def fetch_openneuro_subset(ds_name, subjects):
    dsid = dataset_id(ds_name)
    root = DATA_DIR / dsid
    root.mkdir(parents=True, exist_ok=True)
    for sub in subjects:
        subj_pat = f"sub-{sub}"
        print(f"Downloading {dsid} subject {sub} ... (this may take a while)")
        # Use include filter to keep size manageable: func + anat only
        on.download(dataset=dsid, target_dir=root.as_posix(), include=[subj_pat],
                    insecure=False, overwrite=False)
    return root

ROOT = fetch_openneuro_subset(DATASET, SUBJECTS)
print("Downloaded to:", ROOT)

## 2) Locate BOLD runs and extract ROI time series
We search for `func/*_bold.nii.gz` files, optionally trim initial volumes, and use **Schaefer-200** atlas with a `NiftiLabelsMasker` to extract regional mean signals.

In [ ]:
def find_bold_files(root):
    bolds = sorted(glob.glob(str(root / 'sub-*' / '**' / 'func' / '*_bold.nii*'), recursive=True))
    return bolds

def load_atlas(name='schaefer_200'):
    if name == 'schaefer_200':
        atl = datasets.fetch_atlas_schaefer_2018(n_rois=200, yeo_networks=7, resolution_mm=2)
        labels_img = atl['maps']
    else:
        atl = datasets.fetch_atlas_aal()
        labels_img = atl['maps']
    return labels_img

def extract_timeseries(files, atlas_img, max_files=4, trim_n=5):
    files = files[:max_files]
    masker = input_data.NiftiLabelsMasker(labels_img=atlas_img, standardize=True, detrend=True, t_r=None)
    ts_list, meta = [], []
    for f in files:
        img = image.load_img(f)
        if trim_n:
            img = image.index_img(img, slice(trim_n, None))
        ts = masker.fit_transform(img)  # shape: (n_vols, n_rois)
        ts_list.append(ts)
        meta.append(f)
    return ts_list, meta

bold_files = find_bold_files(ROOT)
if len(bold_files) == 0:
    raise RuntimeError("No BOLD files found. Try another subject or dataset, or remove subject filter.")
atlas_img = load_atlas(ATLAS)
ts_list, meta = extract_timeseries(bold_files, atlas_img, MAX_FILES, TRIM_N_VOLS)
print(f"Found {len(ts_list)} runs. First run shape: {ts_list[0].shape}  (timepoints, ROIs)")

## 3) Chronon wavelets → cross‑scale coupling (zero‑diagonal)
We compute a **CWT** on each ROI time series, convert to power, build a scale×scale correlation per run and ROI, then average across ROIs and runs.

In [ ]:
def cwt_power(ts_1d, n_scales, wavelet):
    scales = np.geomspace(2, 64, num=n_scales)
    coeffs, _ = pywt.cwt(ts_1d, scales, wavelet)
    power = np.abs(coeffs) ** 2
    return power, scales

def empirical_cross_scale_from_runs(ts_list, n_scales=N_SHELLS, wavelet=WAVELET_NAME):
    C_accum = np.zeros((n_scales, n_scales), dtype=float)
    count = 0
    for ts in ts_list:
        # ts: (T, R)
        T, R = ts.shape
        for r in range(R):
            power, scales = cwt_power(ts[:, r], n_scales, wavelet)
            Rmat = np.corrcoef(power)
            C_accum += np.nan_to_num(Rmat)
            count += 1
    C_emp = C_accum / max(1, count)
    np.fill_diagonal(C_emp, 0.0)
    return C_emp, scales

C_emp, scales = empirical_cross_scale_from_runs(ts_list, N_SHELLS, WAVELET_NAME)
print("C_emp shape:", C_emp.shape)

## 4) Fit TSF kernel and estimate \(\lambda\)

In [ ]:
def estimate_lambda(scales):
    i = np.arange(len(scales)); y = np.log(scales)
    A = np.vstack([np.ones_like(i), i]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(np.exp(b))

def tsf_kernel(params, n_shells):
    c0, beta, gamma, aord = params
    idx = np.arange(n_shells)
    D = np.abs(idx[:,None] - idx[None,:])
    Ord = np.log2(1.0 + D)
    K = c0 / (1.0 + beta * (D**np.maximum(gamma, 1e-9))) * np.exp(-aord * Ord)
    np.fill_diagonal(K, 0.0)
    return K

def fit_tsf(C_emp):
    n = C_emp.shape[0]
    def loss(p):
        return np.linalg.norm(C_emp - tsf_kernel(p, n), ord='fro')**2
    x0 = np.array([0.5, 0.5, 1.0, 0.2])
    bounds = [(0,None),(0,None),(0,None),(0,None)]
    res = minimize(loss, x0, bounds=bounds, method='L-BFGS-B')
    return res.x, tsf_kernel(res.x, n), res.fun

lam_hat = estimate_lambda(scales)
params, C_fit, loss_val = fit_tsf(C_emp)
c0, beta, gamma, aord = params
print(f"Estimated lambda: {lam_hat:.4f}")
print("Fitted params:", {"c0":c0, "beta":beta, "gamma":gamma, "alpha_ord":aord})
print(f"Fit loss: {loss_val:.6f}")

mask = ~np.eye(C_emp.shape[0], dtype=bool)
tsf_score = np.corrcoef(C_emp[mask], C_fit[mask])[0,1]
print(f"TSF score (corr off-diag): {tsf_score:.3f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(C_emp, origin='lower', aspect='auto'); plt.title('Empirical C'); plt.colorbar()
plt.subplot(1,2,2); plt.imshow(C_fit, origin='lower', aspect='auto'); plt.title('TSF Kernel Fit'); plt.colorbar()
plt.tight_layout()

# Save
dsid = {'LSD':'ds003059','PSILO':'ds006072','KETAMINE':'ds005917'}[DATASET]
dout = OUT / dsid
dout.mkdir(parents=True, exist_ok=True)
plt.savefig(dout/'empirical_vs_tsf_fit.png', dpi=160)
np.save(dout/'C_emp.npy', C_emp)
np.save(dout/'C_fit.npy', C_fit)
with open(dout/'tsf_params.json','w') as f:
    json.dump({"lambda":lam_hat, "c0":float(c0), "beta":float(beta), "gamma":float(gamma),
               "alpha_ord":float(aord), "loss":float(loss_val), "tsf_score":float(tsf_score),
               "files": meta}, f, indent=2)
print("Saved outputs to:", dout.as_posix())

## 5) Optional: Attempt simple **condition grouping**
Some datasets encode conditions (e.g., `LSD` vs `Placebo`, `Psilocybin` vs `Placebo`, `Ketamine` vs `Saline`) in filenames.
This section naively groups files by substrings; if nothing is found, it will skip gracefully.


In [ ]:
def condition_from_path(p):
    name = os.path.basename(p).lower()
    if 'lsd' in name: return 'LSD'
    if 'placebo' in name or 'pla' in name: return 'Placebo'
    if 'psilocybin' in name or 'psilo' in name: return 'Psilocybin'
    if 'ketamine' in name or 'keta' in name: return 'Ketamine'
    if 'saline' in name or 'control' in name: return 'Control'
    return 'Unknown'

cond_map = {m: condition_from_path(m) for m in meta}
print("Condition map:", cond_map)

# Example: compute a per-run TSF score and compare groups if both exist
def run_tsf_score_per_file(files, atlas_img):
    scores = {}
    for f in files:
        ts = input_data.NiftiLabelsMasker(labels_img=atlas_img, standardize=True, detrend=True).fit_transform(
            image.index_img(image.load_img(f), slice(TRIM_N_VOLS, None))
        )
        C_emp_i, scales_i = empirical_cross_scale_from_runs([ts], N_SHELLS, WAVELET_NAME)
        params_i, C_fit_i, _ = fit_tsf(C_emp_i)
        mask = ~np.eye(C_emp_i.shape[0], dtype=bool)
        score = float(np.corrcoef(C_emp_i[mask], C_fit_i[mask])[0,1])
        scores[f] = score
    return scores

per_file_scores = run_tsf_score_per_file(meta, atlas_img)
df = pd.DataFrame({"file": list(per_file_scores.keys()), "score": list(per_file_scores.values()),
                  "cond": [cond_map[m] for m in meta]})
display(df)

df.to_csv(dout/'per_file_tsf_scores.csv', index=False)
print("Saved per-file TSF scores to:", (dout/'per_file_tsf_scores.csv').as_posix())

### Done
You now have: TSF parameters, coupling matrices, figures, and optional per-run TSF scores.
Scale up by adding subjects or increasing MAX_FILES.